# AATA Detection — External Validation

This notebook evaluates the pretrained `KneeClassifier` on an independent
external test set from UNIFESP. It produces study-level predictions,
comprehensive metrics with 95% bootstrap confidence intervals,
and Grad-CAM visualizations for misclassified cases.

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
IMAGE_FOLDER = "/path/to/JPEG_ext-testTOTAL"   # preprocessed JPEG slices for external test
METADATA_CSV = "/path/to/ext_testTOTAL.csv"    # slice-level metadata for external test
WEIGHTS_PATH = "weights/model.pth"             # trained model checkpoint
GRADCAM_DIR  = "outputs/gradcam/external"      # output directory for Grad-CAM PDFs

# ── Normalization ───────────────────────────────────────────────────────────
# Precomputed from the external test set (mean and std differ slightly from
# the internal training set due to scanner and protocol heterogeneity).
PIXEL_MEAN = 0.0035
PIXEL_STD  = 0.0330

# ── Evaluation ──────────────────────────────────────────────────────────────
THRESHOLD   = 0.1717  # derived from internal validation set (see 01_training.ipynb)
N_BOOTSTRAP = 1000
SEED        = 42

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE      = "cuda" if __import__('torch').cuda.is_available() else "cpu"
NUM_WORKERS = 16

## Imports

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.dataset import MRISliceDataset, build_transforms
from src.metrics import (
    bootstrap_metrics,
    compute_metrics,
    plot_roc,
    print_metrics_table,
)
from src.model import KneeClassifier, generate_gradcam

torch.manual_seed(SEED)
device = torch.device(DEVICE)

## Data

In [ ]:
ext_test = pd.read_csv(METADATA_CSV)

# ── Data quality corrections applied after manual review ────────────────────
# One study was reclassified from AATA-positive to negative after consensus review.
ext_test.loc[
    ext_test["StudyInstanceUID"] == "1.2.826.0.1.3680043.10.474.739111.334", "grupo"
] = 0

# One patient had two studies; the duplicate (patella series only) is excluded.
ext_test = ext_test[ext_test["StudyInstanceUID"] != "1.2.826.0.1.3680043.10.474.739111.5967"]

# Exclude the patella series from a study that retained a valid knee series.
ext_test = ext_test[
    ext_test["SeriesInstanceUID"] != "1.2.826.0.1.3680043.10.474.739111.5913"
]

# Exclude a study containing only a patella series (no axial knee sequences).
ext_test = ext_test[
    ext_test["StudyInstanceUID"] != "1.2.840.113845.11.1000000001956782889.20190927105427.3074505"
]

# Exclude a study with uncertain classification after radiologist review.
ext_test = ext_test[
    ext_test["StudyInstanceUID"] != "1.2.826.0.1.3680043.10.474.660582.277840"
]

print(f"External test slices: {len(ext_test):,}")
print(f"External test studies: {ext_test['StudyInstanceUID'].nunique():,}")

In [ ]:
transform_test = build_transforms(PIXEL_MEAN, PIXEL_STD, augment=False)

ext_test_loader = DataLoader(
    MRISliceDataset(IMAGE_FOLDER, ext_test, label_col="grupo", transform=transform_test),
    batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
)

## Inference

In [ ]:
model = KneeClassifier(num_classes=1).to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

records = []
with torch.no_grad():
    for images, _, sop_uids in tqdm(ext_test_loader, desc="External inference"):
        probs = torch.sigmoid(model(images.to(device))).cpu().numpy()
        for uid, prob in zip(sop_uids, probs):
            records.append({"SOPInstanceUID": uid, "Probability": float(prob)})

sop_results = pd.DataFrame(records).merge(ext_test, on="SOPInstanceUID")

In [ ]:
# Aggregate slice-level probabilities to study level by averaging.
study_ext = (
    sop_results.groupby("StudyInstanceUID")
    .agg(Avg_Probability=("Probability", "mean"), grupo=("grupo", "first"))
    .reset_index()
)
study_ext["Predicted"] = (study_ext["Avg_Probability"] >= THRESHOLD).astype(int)

y_true  = study_ext["grupo"].values
y_pred  = study_ext["Predicted"].values
y_score = study_ext["Avg_Probability"].values

## Evaluation

In [ ]:
point_est = compute_metrics(y_true, y_pred)
ci         = bootstrap_metrics(y_true, y_pred, n_bootstraps=N_BOOTSTRAP, seed=SEED)

print(f"Threshold: {THRESHOLD}  (derived from internal validation set)\n")
print("External Test Set — Point Estimates and 95% Bootstrap Confidence Intervals\n")
print_metrics_table(point_est, ci)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted Neg", "Predicted Pos"],
    yticklabels=["True Neg", "True Pos"],
    ax=ax,
)
ax.set_title("Confusion Matrix — External Test")
plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/confusion_matrix_external.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_roc(y_true, y_score, title="ROC Curve — External Test",
         save_path="outputs/roc_external.png")
plot_roc(y_true, y_score, title="ROC Curve — External Test (zoomed)",
         zoom=True, save_path="outputs/roc_external_zoom.png")

## Grad-CAM Visualization

In [ ]:
# Visualize Grad-CAM for all false-negative studies and the 5 false-positive
# studies with the highest predicted probability.
fn_studies = study_ext[(study_ext["grupo"] == 1) & (study_ext["Predicted"] == 0)]
fp_studies = study_ext[(study_ext["grupo"] == 0) & (study_ext["Predicted"] == 1)]

gradcam_studies = pd.concat([
    fn_studies,
    fp_studies.nlargest(5, "Avg_Probability"),
]).drop_duplicates(subset="StudyInstanceUID")

print(f"Studies selected for Grad-CAM: {len(gradcam_studies)} "
      f"({len(fn_studies)} FN, {min(len(fp_studies), 5)} FP)")

In [ ]:
os.makedirs(GRADCAM_DIR, exist_ok=True)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

target_layer  = model.base_model.layer4
selected_sops = sop_results[
    sop_results["StudyInstanceUID"].isin(gradcam_studies["StudyInstanceUID"])
]

for study_uid in tqdm(gradcam_studies["StudyInstanceUID"], desc="Grad-CAM"):
    study_sops = selected_sops[selected_sops["StudyInstanceUID"] == study_uid]
    true_label = gradcam_studies.loc[
        gradcam_studies["StudyInstanceUID"] == study_uid, "grupo"
    ].values[0]

    pdf_path = os.path.join(GRADCAM_DIR, f"{study_uid}.pdf")
    with PdfPages(pdf_path) as pdf:
        for _, row in study_sops.iterrows():
            image_path = os.path.join(IMAGE_FOLDER, f"{row['SOPInstanceUID']}.jpeg")
            image = Image.open(image_path).convert("RGB")
            input_tensor = transform_test(image.convert("L")).unsqueeze(0)
            input_tensor = torch.from_numpy(
                np.repeat(input_tensor.numpy(), 3, axis=1)
            ).to(device)

            cam = generate_gradcam(model, input_tensor, target_layer)
            cam_resized = cv2.resize(cam, (image.width, image.height), interpolation=cv2.INTER_CUBIC)

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.imshow(image, cmap="gray")
            ax.imshow(cam_resized, cmap="jet", alpha=0.45)
            ax.set_title(
                f"True: {true_label} | Prob: {row['Probability']:.3f}",
                fontsize=13,
            )
            ax.axis("off")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

print(f"Grad-CAM PDFs saved to {GRADCAM_DIR}")